# 02 — Train the classifier (served model + challengers)

**Served model:** calibrated **logistic regression** over word (1–2) + char (3–5) TF-IDF,
`class_weight='balanced'`. Linear on purpose: a feature's contribution to the score is
exactly `coefficient × tfidf` (log-odds), computable and *true* in Java — no fabricated
explanations for a black box.

**Challengers (shown, not served):** XGBoost, and optional DistilBERT.

**Protocol (the honesty section):**
- Stratified **train/test split before any resampling**. The test set is never resampled.
- No SMOTE: `class_weight='balanced'` handles imbalance; SMOTE on high-dimensional sparse
  TF-IDF fabricates implausible documents. (The brief's rule "never SMOTE the test set" is
  honored trivially — we SMOTE nothing.)
- The operating **threshold is chosen on a calibration split** (precision≥0.90, max recall),
  then all threshold metrics are reported on the **untouched test set**.
- Probabilities are **calibrated** (Platt/sigmoid, `cv='prefit'`) so "87% confident" means it.

> **Kaggle setup:** Internet = On. GPU optional (only speeds up the DistilBERT challenger).

In [ ]:
!pip -q install kagglehub xgboost

In [ ]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
SEED = 42

import kagglehub
DATA_DIR = kagglehub.dataset_download("shivamb/real-or-fake-fake-jobposting-prediction")
csvs = glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)
pref = [f for f in csvs if "fake_job_postings" in os.path.basename(f).lower()]
CSV = pref[0] if pref else csvs[0]
df = pd.read_csv(CSV)
print("Loaded:", CSV)
print("Shape:", df.shape)

In [ ]:
# The product is fed pasted free text, so the served model uses ONLY text columns.
# Structured meta-features (has_company_logo, telecommuting, has_questions) are
# deliberately EXCLUDED: they are predictive in EMSCAD but unavailable/leaky at
# serve time, and using them would create train/serve skew.
TEXT_COLS = ["title", "company_profile", "description", "requirements", "benefits"]

def build_text(frame):
    present = [c for c in TEXT_COLS if c in frame.columns]
    return (frame[present].fillna("").astype(str).agg(" ".join, axis=1)
            .str.replace(r"\s+", " ", regex=True).str.strip())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion

# EXACT TF-IDF config — Java must replicate this precisely (see 04 vocab.json export).
# Two blocks, each L2-normalised INDEPENDENTLY, then concatenated (word || char).
WORD_PARAMS = dict(analyzer="word", ngram_range=(1, 2), lowercase=True,
                   strip_accents="unicode", min_df=3, max_features=40000,
                   sublinear_tf=True, norm="l2", use_idf=True, smooth_idf=True)
CHAR_PARAMS = dict(analyzer="char_wb", ngram_range=(3, 5), lowercase=True,
                   strip_accents="unicode", min_df=3, max_features=40000,
                   sublinear_tf=True, norm="l2", use_idf=True, smooth_idf=True)

def build_vectorizer():
    return FeatureUnion([
        ("word", TfidfVectorizer(**WORD_PARAMS)),
        ("char", TfidfVectorizer(**CHAR_PARAMS)),
    ])

In [ ]:
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score,
                             precision_score, recall_score, confusion_matrix)

def threshold_at_precision(y_true, p, target=0.90):
    # Choose the operating threshold on a NON-test split: the one that reaches
    # precision >= target with the highest recall. Returns (threshold, recall, precision).
    from sklearn.metrics import precision_recall_curve
    prec, rec, thr = precision_recall_curve(y_true, p)
    best = None
    for i in range(len(thr)):
        if prec[i] >= target and (best is None or rec[i] > best[1]):
            best = (float(thr[i]), float(rec[i]), float(prec[i]))
    if best is None:
        # target precision unreachable; fall back to the max-precision threshold.
        i = int(np.argmax(prec[:-1])) if len(thr) else 0
        best = (float(thr[i]) if len(thr) else 0.5, float(rec[i]) if len(thr) else 0.0,
                float(prec[i]) if len(thr) else 0.0)
    return best

def eval_at(y_true, p, t):
    pred = (p >= t).astype(int)
    cm = confusion_matrix(y_true, pred)
    return dict(threshold=float(t),
                precision=float(precision_score(y_true, pred, zero_division=0)),
                recall=float(recall_score(y_true, pred, zero_division=0)),
                f1=float(f1_score(y_true, pred, zero_division=0)),
                tn=int(cm[0, 0]), fp=int(cm[0, 1]), fn=int(cm[1, 0]), tp=int(cm[1, 1]))

In [ ]:
from sklearn.model_selection import train_test_split

# Drop exact-duplicate constructed texts BEFORE splitting. EMSCAD reposts the same
# scam under many company names; identical text in both train and test would inflate
# the headline PR-AUC and recall. (Exact-match guard; identical step in 04.)
_d = pd.DataFrame({"text": build_text(df).values, "y": df["fraudulent"].astype(int).values})
_n0 = len(_d)
_d = _d.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
print(f"Dropped {_n0 - len(_d):,} exact-duplicate texts (leakage guard); {len(_d):,} remain")
text, y = _d["text"].values, _d["y"].values

# Split BEFORE any resampling. Test = 20%, held out completely.
X_tr_text, X_te_text, y_tr, y_te = train_test_split(
    text, y, test_size=0.20, stratify=y, random_state=SEED)

# Carve a calibration split out of TRAIN. The calibrator is fit here AND the operating
# threshold is chosen here; the test set stays untouched by both.
X_fit_text, X_cal_text, y_fit, y_cal = train_test_split(
    X_tr_text, y_tr, test_size=0.25, stratify=y_tr, random_state=SEED)

print(f"fit={len(y_fit):,}  calib={len(y_cal):,}  test={len(y_te):,}")
print(f"fraud/fit={y_fit.sum()}  fraud/calib={y_cal.sum()}  fraud/test={y_te.sum()}")

In [ ]:
# --- Vectorize (fit on the fit split only) ---
vectorizer = build_vectorizer()
Xf = vectorizer.fit_transform(X_fit_text)
Xc = vectorizer.transform(X_cal_text)
Xt = vectorizer.transform(X_te_text)
n_features = Xf.shape[1]
print("TF-IDF feature dimension:", n_features)

In [ ]:
# --- Served model: calibrated logistic regression ---
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

lr = LogisticRegression(class_weight="balanced", solver="liblinear",
                        C=1.0, max_iter=2000, random_state=SEED)
lr.fit(Xf, y_fit)

# Platt/sigmoid calibration on the held-out calib split, keeping `lr` fitted so ONE
# coefficient vector survives for the coef*tfidf contributions. cv='prefit' was removed
# in scikit-learn 1.8; FrozenEstimator (>=1.6) is the replacement, with a fallback.
try:
    from sklearn.frozen import FrozenEstimator
    served = CalibratedClassifierCV(FrozenEstimator(lr), method="sigmoid")
except ImportError:
    served = CalibratedClassifierCV(lr, method="sigmoid", cv="prefit")
served.fit(Xc, y_cal)

p_cal = served.predict_proba(Xc)[:, 1]
p_te = served.predict_proba(Xt)[:, 1]

In [ ]:
# --- Metrics for the served model ---
def summarize(name, y_test, p_test, y_calib, p_calib):
    thr, rec_at, prec_at = threshold_at_precision(y_calib, p_calib, 0.90)
    ev = eval_at(y_test, p_test, thr)
    return {
        "model": name,
        "pr_auc": float(average_precision_score(y_test, p_test)),
        "roc_auc": float(roc_auc_score(y_test, p_test)),
        "recall_at_p90": ev["recall"],
        "precision_at_thr": ev["precision"],
        "f1": ev["f1"],
        "threshold": ev["threshold"],
        "cm": [ev["tn"], ev["fp"], ev["fn"], ev["tp"]],
    }

served_row = summarize("Calibrated LogReg (served)", y_te, p_te, y_cal, p_cal)
print(json.dumps(served_row, indent=2))
print("\nConfusion matrix @ chosen threshold [tn, fp, fn, tp]:", served_row["cm"])

In [ ]:
# --- Challenger: XGBoost ---
import xgboost as xgb
spw = (y_fit == 0).sum() / max((y_fit == 1).sum(), 1)
xgb_clf = xgb.XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.1, subsample=0.9,
    colsample_bytree=0.9, tree_method="hist", eval_metric="aucpr",
    scale_pos_weight=spw, random_state=SEED, n_jobs=-1)
xgb_clf.fit(Xf, y_fit)
xp_cal = xgb_clf.predict_proba(Xc)[:, 1]
xp_te = xgb_clf.predict_proba(Xt)[:, 1]
xgb_row = summarize("XGBoost (challenger)", y_te, xp_te, y_cal, xp_cal)
print(json.dumps(xgb_row, indent=2))

In [ ]:
# --- Optional challenger: DistilBERT (skips cleanly without a GPU) ---
distilbert_row = None
RUN_DISTILBERT = True
try:
    import torch
    if not torch.cuda.is_available():
        print("No GPU detected -> skipping DistilBERT challenger (optional).")
        RUN_DISTILBERT = False
except Exception as e:
    print("torch unavailable -> skipping DistilBERT:", e)
    RUN_DISTILBERT = False

if RUN_DISTILBERT:
    try:
        from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                                   TrainingArguments, Trainer)
        from datasets import Dataset
        tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

        def to_ds(texts, labels):
            d = Dataset.from_dict({"text": list(texts), "label": [int(v) for v in labels]})
            return d.map(lambda b: tok(b["text"], truncation=True, max_length=256), batched=True)

        ds_fit, ds_te = to_ds(X_fit_text, y_fit), to_ds(X_te_text, y_te)
        model = AutoModelForSequenceClassification.from_pretrained(
            "distilbert-base-uncased", num_labels=2)
        args = TrainingArguments(output_dir=os.path.join(OUT, "distilbert"),
                                 per_device_train_batch_size=32, num_train_epochs=2,
                                 learning_rate=2e-5, report_to=[], logging_steps=200,
                                 save_strategy="no")
        Trainer(model=model, args=args, train_dataset=ds_fit).train()
        import torch.nn.functional as F
        ds_cal = to_ds(X_cal_text, y_cal)
        pred = Trainer(model=model, args=args)
        dp_cal = F.softmax(torch.tensor(pred.predict(ds_cal).predictions), dim=1)[:, 1].numpy()
        dp_te = F.softmax(torch.tensor(pred.predict(ds_te).predictions), dim=1)[:, 1].numpy()
        # Same protocol as the other rows: threshold@P>=0.90 chosen on calib, evaluated on test,
        # so the "Recall@P>=0.90" column means the same thing for every model.
        distilbert_row = summarize("DistilBERT (challenger)", y_te, dp_te, y_cal, dp_cal)
        print(json.dumps(distilbert_row, indent=2))
    except Exception as e:
        print("DistilBERT challenger failed (optional), continuing:", repr(e))
        distilbert_row = None

In [ ]:
# --- Assemble the metrics table (baseline printed alongside model scores) ---
baseline_row = {
    "model": "Majority-class baseline", "pr_auc": float(y_te.mean()), "roc_auc": 0.5,
    "recall_at_p90": 0.0, "precision_at_thr": 0.0, "f1": 0.0, "threshold": None,
    "cm": [int((y_te == 0).sum()), 0, int((y_te == 1).sum()), 0],
    "accuracy": float((y_te == 0).mean()),
}

rows = [baseline_row, served_row, xgb_row] + ([distilbert_row] if distilbert_row else [])

def fmt(v):
    return "—" if v is None else (f"{v:.4f}" if isinstance(v, float) else str(v))

header = "| Model | PR-AUC | F1 | Recall@P≥0.90 | ROC-AUC | Threshold |\n"
header += "| --- | --- | --- | --- | --- | --- |\n"
lines = []
for r in rows:
    lines.append(f"| {r['model']} | {fmt(r['pr_auc'])} | {fmt(r['f1'])} | "
                 f"{fmt(r['recall_at_p90'])} | {fmt(r['roc_auc'])} | {fmt(r['threshold'])} |")
table = header + "\n".join(lines)
print(table)
print(f"\nReminder — majority-class baseline accuracy on test: {baseline_row['accuracy']:.4%} "
      f"(fraud recall 0.0). Accuracy is NOT a headline metric.")

with open(os.path.join(OUT, "metrics.json"), "w") as f:
    json.dump(rows, f, indent=2)
with open(os.path.join(OUT, "metrics_table.md"), "w", encoding="utf-8") as f:
    f.write(table + "\n")
print("\nSaved metrics.json and metrics_table.md to", OUT)
print(">> Paste metrics_table.md into ml/README.md between the METRICS markers.")

In [ ]:
# --- Calibration curve for the served model (reliability + Brier score) ---
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
import matplotlib.pyplot as plt
frac_pos, mean_pred = calibration_curve(y_te, p_te, n_bins=10, strategy="quantile")
brier = brier_score_loss(y_te, p_te)
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], "--", color="gray", label="perfectly calibrated")
plt.plot(mean_pred, frac_pos, "o-", color="#7c6bff", label=f"served (Brier={brier:.4f})")
plt.xlabel("mean predicted probability"); plt.ylabel("observed fraud fraction")
plt.title("Calibration — served model"); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT, "calibration_served.png"), dpi=120); plt.show()
print("Brier score (lower is better):", round(brier, 4))

In [ ]:
# --- Persist served artifacts for inspection (04 re-fits independently) ---
import joblib
joblib.dump(vectorizer, os.path.join(OUT, "vectorizer.joblib"))
joblib.dump(lr, os.path.join(OUT, "lr.joblib"))
joblib.dump(served, os.path.join(OUT, "served_calibrated.joblib"))
print("Saved vectorizer.joblib, lr.joblib, served_calibrated.joblib to", OUT)

In [ ]:
# --- Export the held-out predictions for the /model page ---------------------------------
# The backend's threshold slider recomputes precision, recall, "real jobs blocked" (FP) and
# "scams let through" (FN) from THESE rows. We export the untouched TEST split only — the exact
# (y_true, calibrated y_score) pairs the served model produced. Nothing on /model is invented;
# it all traces back to this file. Load into Postgres after the V4 migration, like known_scams:
#     psql "$DATABASE_URL" -f validation_predictions_seed.sql
#
# MODEL_NAME/MODEL_VERSION must match the active row seeded in V2__seed_reference_data.sql, so
# the predictions attach to the served model the site is actually running.
MODEL_NAME = "served-logreg-word-char-tfidf"
MODEL_VERSION = "0.1.0"

yt = np.asarray(y_te).astype(int)
ys = np.clip(np.asarray(p_te).astype(float), 0.0, 1.0)
assert len(yt) == len(ys)

# Portable CSV (for any loader / re-analysis).
pd.DataFrame({"y_true": yt, "y_score": ys}).to_csv(
    os.path.join(OUT, "validation_predictions.csv"), index=False)

# SQL seed: batched INSERT ... SELECT that resolves the model_version_id by name+version, so the
# file is portable across databases (no hard-coded id). One statement per batch.
BATCH = 1000
sql_path = os.path.join(OUT, "validation_predictions_seed.sql")
with open(sql_path, "w", encoding="utf-8") as f:
    f.write("-- Held-out TEST-split predictions for the served model (ml/notebooks/02).\n")
    f.write("-- Load after V4__phase7_model_and_reports.sql. Every /model number derives from these rows.\n")
    f.write(f"-- rows={len(yt)}  positives={int(yt.sum())}  negatives={int((yt==0).sum())}\n\n")
    for start in range(0, len(yt), BATCH):
        end = min(start + BATCH, len(yt))
        values = ", ".join(f"({int(yt[i])}, {ys[i]:.6f})" for i in range(start, end))
        f.write(
            "INSERT INTO validation_predictions (model_version_id, y_true, y_score, split)\n"
            "SELECT mv.id, t.y_true, t.y_score, 'TEST'\n"
            f"FROM model_versions mv, (VALUES {values}) AS t(y_true, y_score)\n"
            f"WHERE mv.name = '{MODEL_NAME}' AND mv.version = '{MODEL_VERSION}';\n"
        )

print(f"Saved validation_predictions.csv ({len(yt)} rows) and validation_predictions_seed.sql to", OUT)
print(f"  positives (fraud)={int(yt.sum())}  negatives (legit)={int((yt==0).sum())}  "
      f"prevalence={yt.mean():.4f}")
print(">> Copy validation_predictions_seed.sql into the deploy and load it after Flyway V4.")


### Reading the table like an interviewer

- **PR-AUC** is the primary metric; the **majority-class baseline PR-AUC equals the
  prevalence (~0.048)** — the honest "no-skill" floor.
- **Recall@precision≥0.90** is the number that matters for the product: at a precision high
  enough to avoid defaming real employers, *how many scams do we still catch?* Expect this to
  be well below 1.0 — that is the real, honest tradeoff, made visible on the `/model` page.
- XGBoost/DistilBERT usually score higher PR-AUC but are **challengers**: they cannot produce
  the exact, true `coef×tfidf` explanation the served linear model gives in Java.